# Part 5 — AlphaEarth for all 2,433 patches (2019 and 2018)

**Goal:** `pj_vmi/processed/` holds, for each year, `ae_<year>.npy` (2433, 64, 128, 128) float16 (NaN = no data), `qa_ae_<year>.csv` and `ae_<year>_info.json`, on the same grid as TESSERA; `pj_vmi/results/subpixel_offset.csv` is updated with both embeddings and both years.

**Needs:** the Earth Engine project of Part 4, and Parts 1 and 3 on the Drive. **Runtime:** CPU; keep the tab open. **Time:** about 10 min per year for the download (pilot estimate), a few minutes to merge, 5–15 min to copy to the Drive. **Drive space:** about 10.2 GB for the merged files; the shards (about 10.2 GB) are deleted at the end.

**If the runtime disconnects:** run steps 1–4 again, then the cell that was running; finished shards are skipped.

**Files needed in the repository** (new in this part): `notebooks/data/05_alphaearth_full.ipynb`.

Data attribution: "The AlphaEarth Foundations Satellite Embedding dataset is produced by Google and Google DeepMind." (CC-BY 4.0)

## 1. Runtime, Drive and configuration
Set `GEE_PROJECT` to the ID of your Earth Engine project.

In [ ]:
import sys, shutil, platform, runpy
from pathlib import Path
import json, numpy as np, pandas as pd
print("Python", platform.python_version())
print(f"free space in /content: {shutil.disk_usage('/content').free / 1e9:.0f} GB")

from google.colab import drive
drive.mount("/content/drive")
!df -h /content/drive | tail -1

GEE_PROJECT = "your-project-id"   # <-- EDIT
DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")

PROC = Path("/content/processed")
PROC_DRIVE = DRIVE / "processed"
SHARDS = DRIVE / "shards"
RESULTS = DRIVE / "results"
YEARS = [2019, 2018]

## 2. Code

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt

def run_script(name, *args):
    # Runs scripts/<name>.py inside this kernel, so that it uses the Earth Engine login of step 4.
    sys.argv = [name] + [str(a) for a in args]
    runpy.run_path(str(CODE / "scripts" / f"{name}.py"), run_name="__main__")

## 3. Outputs of Parts 1 and 3

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "instances.npy", "qa_tessera_2019.csv", "qa_tessera_2018.csv"]:
    shutil.copy(PROC_DRIVE / f, PROC / f)
meta = pd.read_csv(PROC / "meta.csv", index_col="idx")
print(len(meta), "patches")

## 4. Earth Engine login
**Done when:** a positive number of AlphaEarth images is printed for each year.

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
for y in YEARS:
    n = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").filterDate(f"{y}-01-01", f"{y + 1}-01-01").size().getInfo()
    print(y, n, "AlphaEarth images")

## 5. Download, one year per cell
Each line `[part k/25]` is one shard of 100 patches written to the Drive.

**Done when:** the cell ends with `done` and every shard line says `errors: 0`. Run the extra `--retry-errors` cell of a year **only** if one of its shards reported errors (otherwise it re-reads all shards for nothing).

**Common errors:** `Too many requests` or quota messages: the script waits and retries by itself; if errors remain, run the year again with `"--workers", 4`.

In [ ]:
run_script("fetch_alphaearth", "--project", GEE_PROJECT, "--processed", PROC, "--year", 2019,
           "--shard-dir", SHARDS / "ae_2019", "--workers", 8)

In [ ]:
# only if some shard of 2019 reported errors
run_script("fetch_alphaearth", "--project", GEE_PROJECT, "--processed", PROC, "--year", 2019,
           "--shard-dir", SHARDS / "ae_2019", "--workers", 8, "--retry-errors")

In [ ]:
run_script("fetch_alphaearth", "--project", GEE_PROJECT, "--processed", PROC, "--year", 2018,
           "--shard-dir", SHARDS / "ae_2018", "--workers", 8)

In [ ]:
# only if some shard of 2018 reported errors
run_script("fetch_alphaearth", "--project", GEE_PROJECT, "--processed", PROC, "--year", 2018,
           "--shard-dir", SHARDS / "ae_2018", "--workers", 8, "--retry-errors")

## 6. Merge the shards
**Done when**, for each year: `25 shards, 2433/2433 patches`, `missing: 0, duplicated: 0, errors: 0`.

In [ ]:
for y in YEARS:
    print(f"===== {y}")
    !python scripts/merge_shards.py --shard-dir "{SHARDS}/ae_{y}" --processed "{PROC}" --name ae_{y}

## 7. Checks per tile and sub-pixel offsets
The first table summarises the AlphaEarth QA per year and tile: patches with no-data pixels, deviation from unit length (`norm_dev`, should stay close to 0.0016 as in the pilot), share of patches whose best shift is (0, 0), median ratio. The second is `subpixel_offset.py` for both embeddings and both years.

**Done when:** both tables are printed and `results/subpixel_offset.csv` is saved on the Drive.

In [ ]:
pd.set_option("display.width", 200)
rows = []
for y in YEARS:
    qa = pd.read_csv(PROC / f"qa_ae_{y}.csv", index_col="idx").join(meta[["TILE"]])
    for tile, g in qa.groupby("TILE"):
        rows.append({"year": y, "tile": tile, "n": len(g),
                     "with_nodata": int((g.nodata_frac > 0).sum()), "max_nodata": round(g.nodata_frac.max(), 3),
                     "norm_dev_median": round(g.norm_dev.median(), 4), "norm_dev_max": round(g.norm_dev.max(), 4),
                     "best_0_0": round(((g.best_dx == 0) & (g.best_dy == 0)).mean(), 3),
                     "ratio_median": round(g["ratio_+0_+0"].median(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
!python scripts/subpixel_offset.py --processed "{PROC}" \
    --qa "{PROC}/qa_tessera_2019.csv" "{PROC}/qa_tessera_2018.csv" "{PROC}/qa_ae_2019.csv" "{PROC}/qa_ae_2018.csv" \
    --out "{RESULTS}/subpixel_offset.csv"

## 8. Copy to the Drive
**Done when:** every line ends with `ok=True`.

In [ ]:
for y in YEARS:
    for f in [f"ae_{y}.npy", f"qa_ae_{y}.csv", f"ae_{y}_info.json"]:
        shutil.copy(PROC / f, PROC_DRIVE / f)
        same = (PROC_DRIVE / f).stat().st_size == (PROC / f).stat().st_size
        print(f"{f:20s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={same}")

## 9. Delete the shards
Deletes the shards of a year only if its merged file is on the Drive with the same size as the local one. Deleted files go to the Drive trash and still count against the quota until the trash is emptied.

In [ ]:
for y in YEARS:
    f = f"ae_{y}.npy"
    ok = (PROC_DRIVE / f).exists() and (PROC_DRIVE / f).stat().st_size == (PROC / f).stat().st_size
    if ok and (SHARDS / f"ae_{y}").exists():
        shutil.rmtree(SHARDS / f"ae_{y}")
        print("deleted", SHARDS / f"ae_{y}")
    else:
        print("kept", SHARDS / f"ae_{y}", "" if ok else "(merged file not verified on the Drive)")
!df -h /content/drive | tail -1

## Results to report
1. The commit line of step 2 and the output of step 4.
2. For each year, the last lines of the download cell and the full output of the merge (step 6).
3. Both tables of step 7.
4. The `ok=` lines of step 8 and the output of step 9.
5. Any error message, in full.